# Mutability & Frozen Models

By default, Pydantic model instances are **mutable**. You can change the values of their fields after the object has been instantiated. However, in strict functional programming paradigms or specific architectural setups, you may want to ensure that a model instance cannot be changed once it is created.

## 1. The Default Behavior (Mutable)

```python
from pydantic import BaseModel

class MutableModel(BaseModel):
    field: int

m1 = MutableModel(field=10)
m1.field = 20 # This works perfectly fine!
print(m1.field) # 20

```

## 2. Making a Model Immutable (`frozen=True`)

You can lock down an entire model by setting `frozen=True` inside the `model_config`. (Note: Pydantic borrows this terminology from Python's standard `dataclasses`).

```python
from pydantic import ConfigDict, ValidationError

class FrozenModel(BaseModel):
    model_config = ConfigDict(frozen=True)
    field: int

m2 = FrozenModel(field=10)

try:
    m2.field = 20
except ValidationError as e:
    print(e)
    # Output: 1 validation error for FrozenModel
    # field: Instance is frozen

```

Attempting to modify a frozen instance will instantly raise a `ValidationError`.

## 3. The Core Python Implication: Hashability

Beyond just preventing accidental data mutation, freezing a model has a profound impact on how the object interacts with core Python data structures.

In Python, you can only use **Hashable** objects as keys in a dictionary or as elements in a `set`. By definition, mutable objects (like standard lists, dicts, and default Pydantic models) are unhashable. If an object can change its state, its memory hash could theoretically change, making dictionary lookups impossible.

Because `frozen=True` guarantees the object's state will never change, Python allows it to be hashed.

```python
# ❌ THIS FAILS (Mutable models are unhashable)
try:
    my_dict = {m1: "Some Value"} 
except TypeError as e:
    print(f"TypeError: {e}") # TypeError: unhashable type: 'MutableModel'

# ✅ THIS WORKS (Frozen models are hashable)
my_dict = {m2: "Some Value"}
print(my_dict[m2]) # "Some Value"

```

### Interview Preparation: Mutability & Hashability

<details>
<summary><b>1. You are building an in-memory caching system where the cache keys are Pydantic models representing user queries, and the cache values are the API responses. If you try to implement this with a standard Pydantic model, it crashes. Why, and how do you fix it?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
It crashes with a `TypeError: unhashable type` because standard Pydantic models are mutable by default. In Python, dictionary keys (or cache keys) must be hashable, meaning their state cannot change after creation. <br><br>
To fix this, I would set `model_config = ConfigDict(frozen=True)` on the Pydantic model representing the user query. This makes the model immutable and therefore hashable, allowing it to be safely used as a dictionary or cache key.
</details>

<details>
<summary><b>2. You review a PR where a developer set `model_config = ConfigDict(frozen=True, validate_assignment=True)`. What is wrong with this configuration?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
These two configuration settings are logically contradictory. <br><br>
`frozen=True` makes the instance entirely immutable; any attempt to assign a new value to a field will raise an error stating the instance is frozen. <br><br>
`validate_assignment=True` tells Pydantic to run type validation *when* a field is reassigned. Because the model is frozen, a reassignment can never happen in the first place, rendering `validate_assignment=True` completely dead code. You should use one or the other depending on the desired architecture, but never both.
</details>

<details>
<summary><b>3. Is an exception raised by attempting to modify a frozen Pydantic model a standard Python `TypeError`, or a Pydantic `ValidationError`? Why does this distinction matter?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
It raises a Pydantic `ValidationError`. <br><br>
This distinction matters deeply for API error handling (like in FastAPI). If it raised a standard Python exception, it might bubble up as an unhandled Internal Server Error (HTTP 500). Because it raises a `ValidationError`, FastAPI knows exactly how to catch it and format it into a structured HTTP 422 Unprocessable Entity response, keeping the API contract intact.
</details>